# Avance 2

### Conectarse a la API de Yelp para obtener información de negocios locales según la ciudad seleccionada.

In [1]:
#Importamos las librerías que se necesitan en este avance
import pandas as pd
import numpy as np
import requests
#requests: permite enviar solicitudes HTTP a Yelp.
#Pandas nos permite trabajar en dataframes

In [2]:
# La API key está en el archivo API_Key.txt (ignorado por git para no subirla a GitHub)
with open("API_Key.txt", encoding="utf-8") as archivo:
    api_key = archivo.read().strip()

In [3]:
# Endpoint de Yelp para buscar negocios
url = "https://api.yelp.com/v3/businesses/search"

# Autenticación: enviamos la API key en el encabezado
headers = {"Authorization": f"Bearer {api_key}"}

In [4]:
# Ciudad elegida para buscar negocios en Yelp
ciudad = "Chicago"

In [5]:
# Extraemos 240 restaurantes (máximo de Yelp por búsqueda) en 6 bloques de 40
negocios = []

for i in range(0, 240, 40):
    params = {"location": 'chicago', "categories": "restaurants", "limit": 40, "offset": i}
    resp = requests.get(url, headers=headers, params=params, timeout=30)
    resp.raise_for_status()   # detiene el código si Yelp devuelve un error
    negocios.extend(resp.json()["businesses"])

print("Negocios obtenidos:", len(negocios))

Negocios obtenidos: 240


In [6]:
columnas = ["nombre", "categoria", "categorias", "rating", "reseñas", "precio"]
datos = []

for negocio in negocios:
    nombre = negocio["name"]
    lista_categorias = [c["title"] for c in negocio["categories"]]
    categoria = lista_categorias[0]             # categoría principal
    categorias = ", ".join(lista_categorias)    # todas las categorías
    rating = negocio["rating"]
    reseñas = negocio["review_count"]
    precio = negocio.get("price")               # puede no existir
    datos.append([nombre, categoria, categorias, rating, reseñas, precio])

df_yelp = pd.DataFrame(datos, columns=columnas)
df_yelp["ciudad"] = ciudad
df_yelp.shape

(240, 7)

In [7]:
df_yelp.info()

<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   nombre      240 non-null    str    
 1   categoria   240 non-null    str    
 2   categorias  240 non-null    str    
 3   rating      240 non-null    float64
 4   reseñas     240 non-null    int64  
 5   precio      185 non-null    str    
 6   ciudad      240 non-null    str    
dtypes: float64(1), int64(1), str(5)
memory usage: 13.3 KB


#### Exploración de los datos de Yelp
- Se obtuvieron 240 restaurantes de Chicago (máximo por búsqueda), sin duplicados.
  Yelp informa 7.900, por lo que la muestra (los más relevantes) no representa toda la ciudad.
- La calificación es alta y homogénea (mediana 4,4; rango 3,4 a 5,0).
- Las reseñas están muy sesgadas (mediana 326,5; máximo 10.761), por lo que se usa la mediana.
- El 27,5% no tiene precio y se marcó como "No especificado". El precio más frecuente es `$$` (47,1%).

In [8]:
# Porcentaje de precios faltantes
print(round(df_yelp["precio"].isna().mean() * 100, 1), "%")

22.9 %


In [9]:
# Duplicados: filas completas y nombres repetidos
print("Filas duplicadas:", df_yelp.duplicated().sum())
print("Nombres repetidos:", df_yelp["nombre"].duplicated().sum())
df_yelp[df_yelp["nombre"].duplicated(keep=False)].sort_values("nombre")

Filas duplicadas: 0
Nombres repetidos: 1


,nombre,categoria,categorias,rating,reseñas,precio,ciudad
12,Qing Xiang Yuan Dumplings,Chinese,"Chinese, Barbeque",4.4,2221,$$,Chicago
115,Qing Xiang Yuan Dumplings,Dim Sum,"Dim Sum, Barbeque, Coffee & Tea",4.6,36,NaN,Chicago


In [10]:
# El precio nulo no se imputa: se marca como "No especificado"
df_yelp["precio"] = df_yelp["precio"].fillna("No especificado")
df_yelp.isna().sum()

nombre        0
categoria     0
categorias    0
rating        0
reseñas       0
precio        0
ciudad        0
dtype: int64

In [11]:
#Revisamos los datos de los dataframe   
print(df_yelp[["rating", "reseñas"]].describe())
print((df_yelp["precio"].value_counts(normalize=True) * 100).round(1))

           rating       reseñas
count  240.000000    240.000000
mean     4.317083    823.708333
std      0.255647   1404.175160
min      3.400000      8.000000
25%      4.200000    178.250000
50%      4.300000    396.000000
75%      4.500000    822.000000
max      4.900000  10765.000000
precio
$$                 46.7
No especificado    22.9
$$$                21.7
$$$$                6.7
$                   2.1
Name: proportion, dtype: float64


In [12]:
# Creamos unna fila por restaurante y categoría
df_cat = pd.json_normalize(negocios, record_path="categories", meta=["name"])
print("Categorías distintas:", df_cat["title"].nunique())
df_cat["title"].value_counts().head(40)

Categorías distintas: 118


title
Cocktail Bars            60
New American             39
Italian                  33
Bars                     30
Breakfast & Brunch       26
Mediterranean            25
Seafood                  25
American                 25
Sushi Bars               22
Asian Fusion             21
Thai                     18
Pizza                    17
Japanese                 16
Steakhouses              14
Korean                   14
Ramen                    13
Burgers                  12
Mexican                  12
Wine Bars                11
Chinese                  11
Barbeque                 11
Noodles                  11
Sandwiches               11
Tapas/Small Plates       10
Latin American            9
Desserts                  8
Venues & Event Spaces     8
Lounges                   8
Beer Bar                  7
Soup                      7
Izakaya                   7
Bakeries                  6
Salad                     6
Hot Pot                   6
Cafes                     6
French        

In [14]:
#Veamos los keys del diccionario recibido
datos_api.keys()

NameError: name 'datos_api' is not defined

In [ ]:
datos_api['businesses']
#Aqui inspeccionamos el restaurante

In [ ]:
df = pd.json_normalize(
   negocios,
    record_path='categories',
    meta=[
        'name',
        'review_count',
        'rating',
        'transactions',
        'price',
        'location.city',
        'phone'
    ],
    sep='_',
    errors='ignore'
)

df

In [ ]:
#El primer elemento del diccionario indica el total de restaurants existentes en la API
print(f"Yelp informa {datos_api['total']} resultados para la búsqueda de restaurantes en Chicago.")

### Extraer datos como nombre del negocio, categoría, calificación, número de reseñas, calificación, etc. 

In [ ]:
#Aqui inspeccionamos el restaurante 
restaurante = datos_api['businesses'][0]
#Esto mostrará los campos disponibles para el primer restaurante.
restaurante.keys()

In [ ]:
# Creamos una lista para guardar los negocios obtenidos en cada solicitud
negocios = []

# Creamos un bucle para extraer los datos recorriendo cada página para tratar de obtener un total de 200 datos 
for offset in range(0, 200, 50):

#Enviamos la solicitud y la conviertimos la respuesta JSON en un diccionario de Python.
    params = {
        "location": "Chicago",
        "categories": "restaurants",
        "limit": 50,
        "offset": offset
    }

    # Nueva solicitud a Yelp
    resp = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=10
    )

    # Convertimos la respuesta a JSON
    datos_api = resp.json()

    print(resp.status_code)

    #Agregamos a negocios cada elemento de la lista recibida. Asi guardamos los restaurantes de las distintas páginas en una sola lista.
    negocios.extend(datos_api["businesses"])

In [ ]:
#revisamos los datos 
negocios[0]

In [ ]:
# 1 Definimos las columnas
columnas = ["nombre", "categoria", "categorias", "rating", "reseñas", "precio"]

# 2 Creamos una lista vacía
datos = []

# 3 Recorremos los 200 negocios
for negocio in negocios:
    # 4 Extraemos los datos
    nombre = negocio["name"]
    lista_categorias = [c["title"] for c in negocio["categories"]]
    categoria = lista_categorias[0]               # categoría principal
    categorias = ", ".join(lista_categorias)      # todas las categorías, separadas por coma
    rating = negocio["rating"]
    reseñas = negocio["review_count"]
    precio = negocio.get("price")

    # 5 Guardamos los datos
    datos.append([nombre, categoria, categorias, rating, reseñas, precio])

# 6 Creamos el DataFrame
df_yelp = pd.DataFrame(datos, columns=columnas)

# 7 Visualizamos las primeras filas
df_yelp.head()

In [ ]:
#Vemos como están los datos 
df_yelp.info()
#Podemos ver que la información nos muestra cuatro columnas en las cuales no existen nulos pero si existe 46 datos faltantes en precio

In [ ]:
#Aqui ve el porcentaje del promedio de valores faltantes
df_yelp["precio"].isnull().mean() * 100
#El 24 % de los restaurantes carece de información de precio.

In [ ]:
#Vamos a verificar como esta el dataframe
df_yelp.shape
#Tenemos 200 filas y 6 columnas

In [ ]:
#Vamos a comprobar si es que hay duplicados
print('Filas duplicadas:', df_yelp.duplicated().sum())
print('Nombres repetidos:', df_yelp['nombre'].duplicated().sum())
df_yelp[df_yelp['nombre'].duplicated(keep=False)].sort_values('nombre')

In [ ]:
#Aquí vemos que no hay valores nulos
df_yelp['precio'] = df_yelp['precio'].fillna('No especificado')
df_yelp['ciudad'] = 'Chicago'
df_yelp.isna().sum()

In [ ]:
#Ahora vemos los porcentajes de cada precio
df_yelp['precio'].value_counts(normalize=True) * 100
#Tenemos que el 55% de los restaurantes es `$$`; solo 3.5 son `$` y 3 son `$$$$`. el 24% no tiene dato de precio.

In [ ]:
df_yelp['categoria'].value_counts(normalize=True).head(15)*100

In [ ]:
#Aqui vemos las categorias unicas
df_yelp['categoria'].nunique()
#- Los 200 restaurantes tienen 86 categorías distintas, por lo que se agruparán en familias más amplias 
# para poder compararlas con las preferencias alimenticias de los clientes.

#### Exploración de los datos de Yelp
- Se obtuvieron 200 restaurantes de Chicago, sin duplicados.
- La calificación es alta y homogénea (mediana 4,4; rango 3,6 a 5,0).
- Las reseñas tienen una distribución muy sesgada (mediana 385, máximo 10.451), por lo que se usa la mediana

### Integrar esta información al análisis que se está realizando para una ciudad  en específico. 